# 朝代预测数据集：开发日志

> 本 Notebook 记录 Mendeley 明清陶瓷数据集的 API 查询与下载全过程，含逐步注释与实际运行结果。
>
> 日期：2026-07-10  
> 作者：Qoder AI 辅助开发


## 一、环境准备

In [ ]:
# 导入所有必要模块
import json          # 解析 Mendeley API 返回的 JSON
import os            # 文件路径与目录操作
import subprocess    # 调用系统 curl 命令（绕过 403）
import sys           # 标准错误输出
import urllib.request  # 发送 HTTP 请求到 Mendeley API

print("✓ 所有模块导入成功")


✓ 所有模块导入成功


## 二、配置参数

Mendeley Data 提供公开 REST API，无需鉴权即可查询数据集文件列表。
格式：`https://data.mendeley.com/api/datasets/{dataset_id}/files`


In [ ]:
# 数据集 ID（来自 DOI: 10.17632/xzg7d8kygy.1）
DATASET_ID = "xzg7d8kygy"

# Mendeley 公开文件列表 API
API_URL = f"https://data.mendeley.com/api/datasets/{DATASET_ID}/files"

# 本地保存目录（相对于仓库根目录）
OUT_DIR = os.path.join(os.path.dirname(os.getcwd()), "datasets", "mendeley_ceramics")
# 若在仓库根目录运行，使用以下路径：
OUT_DIR = os.path.join(os.getcwd(), "..", "datasets", "mendeley_ceramics")
OUT_DIR = os.path.normpath(OUT_DIR)

os.makedirs(OUT_DIR, exist_ok=True)
print(f"数据集 API：{API_URL}")
print(f"本地保存路径：{OUT_DIR}")


数据集 API：https://data.mendeley.com/api/datasets/xzg7d8kygy/files
本地保存路径：/root/xiaohanchen/projects/vps-llm-kaogu/datasets/mendeley_ceramics


## 三、查询 Mendeley 文件列表

调用 API 获取数据集所有文件的元数据，包括文件名、大小和**真实下载 URL**。

> **注意**：直接访问 Mendeley 网页的 "Download All" 会返回 HTML 而非文件，  
> 必须通过 API 获取每个文件的 `content_details.download_url`。


In [ ]:
# 发送 GET 请求到 Mendeley API
# 必须设置 User-Agent，否则某些请求会被拒绝
req = urllib.request.Request(
    API_URL,
    headers={"User-Agent": "Mozilla/5.0"}
)

with urllib.request.urlopen(req, timeout=30) as resp:
    files = json.loads(resp.read())

# 查看返回结构
print(f"API 返回文件数量：{len(files)} 个")
print(f"\n前 3 条记录示例：")
for f in files[:3]:
    print(f"  文件名：{f['filename']}")
    print(f"  大小：{f['size']} bytes ({f['size']/1024:.1f} KB)")
    print(f"  下载URL（截断）：{f['content_details']['download_url'][:80]}...")
    print()


API 返回文件数量：100 个

前 3 条记录示例：
  文件名：M_ming100.jpg
  大小：4946 bytes (4.8 KB)
  下载URL（截断）：https://data.mendeley.com/public-files/datasets/xzg7d8kygy/files/12683e94-d...

  文件名：M_ming101.jpg
  大小：5469 bytes (5.3 KB)
  下载URL（截断）：https://data.mendeley.com/public-files/datasets/xzg7d8kygy/files/bbc30104-6...

  文件名：M_ming102.jpg
  大小：5907 bytes (5.8 KB)
  下载URL（截断）：https://data.mendeley.com/public-files/datasets/xzg7d8kygy/files/662336c2-3...



### 发现：API 仅返回 M_ming（明代梅瓶）

分析 100 个文件名的前缀，发现**公开 API 只暴露了 `M_ming` 类**，共 100 张图像。

数据集描述的 4 个类别：
| 类别标签 | 含义 | API 是否可获取 |
|---------|------|--------------|
| `M_ming` | 明代梅瓶 | ✅ 100 张 |
| `M_qing` | 清代梅瓶 | ❌ 未在 API 中 |
| `Y_ming` | 明代玉壶春瓶 | ❌ 未在 API 中 |
| `Y_qing` | 清代玉壶春瓶 | ❌ 未在 API 中 |

**结论**：单类数据无法做朝代分类，需联系作者获取完整数据集（论文作者：陕西科技大学 Ruirui Zhang）。


In [ ]:
# 统计各类别前缀分布
from collections import Counter

prefixes = Counter(f["filename"].split(".")[0].rsplit("_", maxsplit=1)[0] for f in files)
print("类别分布：")
for prefix, count in sorted(prefixes.items()):
    print(f"  {prefix}: {count} 张")

print(f"\n总计：{sum(prefixes.values())} 张，{len(prefixes)} 个类别")
print("⚠️  仅有 M_ming 一类，无法做朝代分类训练！")


类别分布：
  M_ming: 100 张

总计：100 张，1 个类别
⚠️  仅有 M_ming 一类，无法做朝代分类训练！


## 四、下载脚本

### 问题：urllib 直接请求返回 403

最初尝试用 Python `urllib.request.urlretrieve()` 下载文件，但 Mendeley 的 CDN 返回 **HTTP 403 Forbidden**。

### 解决方案：改用 `curl` + `User-Agent` 头

系统 curl 可以正确跟随重定向并附带正确的 TLS 握手，绕过 CDN 的限制。


In [ ]:
def download_all(files, out_dir, verbose=True):
    """
    逐一下载 Mendeley 数据集文件。
    
    - 使用 curl 绕过 CDN 的 403 限制
    - 跳过已存在且大于 100 字节的文件（断点续传）
    - 小于 100 字节的文件视为下载失败（通常是错误页面 HTML）
    """
    success, skip, fail = 0, 0, 0
    
    for i, f in enumerate(files, 1):
        fname = f["filename"]
        url = f["content_details"]["download_url"]
        out_path = os.path.join(out_dir, fname)
        
        # 跳过已下载的文件
        if os.path.exists(out_path) and os.path.getsize(out_path) > 100:
            if verbose:
                print(f"  [{i:3d}/{len(files)}] {fname} — 已存在，跳过")
            skip += 1
            continue
        
        # 使用 curl -sL（静默 + 跟随重定向）+ User-Agent 绕过 403
        result = subprocess.run(
            ["curl", "-sL", "-H", "User-Agent: Mozilla/5.0", "-o", out_path, url],
            capture_output=True
        )
        
        # 验证下载结果（文件 > 100 字节才算成功）
        if result.returncode == 0 and os.path.exists(out_path) and os.path.getsize(out_path) > 100:
            if verbose:
                size_kb = os.path.getsize(out_path) / 1024
                print(f"  [{i:3d}/{len(files)}] {fname} ✓ ({size_kb:.1f} KB)")
            success += 1
        else:
            print(f"  [{i:3d}/{len(files)}] {fname} ✗ 失败", file=sys.stderr)
            fail += 1
    
    return success, skip, fail


# 执行下载
print(f"开始下载到：{OUT_DIR}")
print("-" * 50)
success, skip, fail = download_all(files, OUT_DIR, verbose=False)
print(f"\n下载完成：成功 {success}，跳过（已存在）{skip}，失败 {fail}")


开始下载到：/root/xiaohanchen/projects/vps-llm-kaogu/datasets/mendeley_ceramics
--------------------------------------------------

下载完成：成功 0，跳过（已存在）100，失败 0


## 五、下载结果验证

In [ ]:
import os

# 验证本地文件
local_files = sorted(f for f in os.listdir(OUT_DIR) if f.endswith(".jpg"))
sizes = [os.path.getsize(os.path.join(OUT_DIR, f)) for f in local_files]

print(f"本地文件数量：{len(local_files)} 张")
print(f"文件大小范围：{min(sizes)/1024:.1f} KB ~ {max(sizes)/1024:.1f} KB")
print(f"平均文件大小：{sum(sizes)/len(sizes)/1024:.1f} KB")
print(f"总计磁盘占用：{sum(sizes)/1024:.0f} KB ({sum(sizes)/1024/1024:.2f} MB)")
print()

# 类别分布
from collections import Counter
labels = [f.rsplit("_", 1)[0] for f in local_files]
dist = Counter(labels)
print("类别分布（朝代标签）：")
for label, cnt in sorted(dist.items()):
    print(f"  {label}: {cnt} 张")
    
print()
print("前 5 个文件：")
for fname in local_files[:5]:
    sz = os.path.getsize(os.path.join(OUT_DIR, fname))
    print(f"  {fname}  ({sz/1024:.1f} KB)")


本地文件数量：100 张
文件大小范围：4.8 KB ~ 9.0 KB
平均文件大小：5.6 KB
总计磁盘占用：563 KB (0.55 MB)

类别分布（朝代标签）：
  M_ming: 100 张

前 5 个文件：
  M_ming100.jpg  (4.8 KB)
  M_ming101.jpg  (5.3 KB)
  M_ming102.jpg  (5.8 KB)
  M_ming103.jpg  (5.9 KB)
  M_ming104.jpg  (5.9 KB)


## 六、总结与下一步

### 本次工作成果

| 产出 | 说明 |
|------|------|
| `notebooks/findings.ipynb` | 6个朝代预测数据集调研报告 |
| `scripts/download_mendeley.py` | Mendeley 数据集下载脚本（可复用） |
| `datasets/mendeley_ceramics/` | M_ming × 100 张图像（明代梅瓶）|

### 技术经验

1. **Mendeley API 用法**：`/api/datasets/{id}/files` → 获取每个文件的真实 `download_url`
2. **403 绕过**：Python `urllib` 会被 CDN 拒绝，改用系统 `curl -sL -H "User-Agent: Mozilla/5.0"` 即可
3. **完整性校验**：下载后检查文件大小 > 100 字节，避免将错误 HTML 页面误认为图像

### 阻塞项 & 下一步

| 优先级 | 任务 | 联系方式 |
|--------|------|---------|
| 🥇 | 申请 Bronze Ding 数据集（4朝代+11时期，CVPR2023）| [email removed] |
| 🥈 | 联系 Mendeley 作者获取完整 4 类陶瓷数据 | DOI: 10.17632/xzg7d8kygy.1 |
| 🥉 | 确认 Nature/Sensors 论文数据可用性 | 论文 Supplementary / Data Availability |
